In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score
)
import joblib

In [19]:
DATA_PATH = r"C:\Users\DELL\Pictures\Documents\CodeBind-projects\LoanApprov\loan_approval_dataset.csv"
TARGET_COL = " loan_status"          # column we want to predict
DROP_COLS = ["Loan_ID"]             # identifier columns, not predictive
RANDOM_STATE = 42

In [20]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df.head())
print("\nMissing values per column:\n", df.isnull().sum())

Shape: (4269, 13)
   loan_id   no_of_dependents      education  self_employed   income_annum  \
0        1                  2       Graduate             No        9600000   
1        2                  0   Not Graduate            Yes        4100000   
2        3                  3       Graduate             No        9100000   
3        4                  3       Graduate             No        8200000   
4        5                  5   Not Graduate            Yes        9800000   

    loan_amount   loan_term   cibil_score   residential_assets_value  \
0      29900000          12           778                    2400000   
1      12200000           8           417                    2700000   
2      29700000          20           506                    7100000   
3      30700000           8           467                   18200000   
4      24200000          20           382                   12400000   

    commercial_assets_value   luxury_assets_value   bank_asset_value  \
0       

In [21]:
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
 


In [22]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())
 

In [23]:
if "Dependents" in df.columns:
    df["Dependents"] = df["Dependents"].astype(str).str.replace("+", "", regex=False)
    df["Dependents"] = pd.to_numeric(df["Dependents"], errors="coerce").fillna(0)
 

In [24]:
label_encoders = {}
categorical_cols = df.select_dtypes(include="object").columns.tolist()
if TARGET_COL in categorical_cols:
    categorical_cols.remove(TARGET_COL)
 
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [25]:
df.columns 

Index(['loan_id', ' no_of_dependents', ' education', ' self_employed',
       ' income_annum', ' loan_amount', ' loan_term', ' cibil_score',
       ' residential_assets_value', ' commercial_assets_value',
       ' luxury_assets_value', ' bank_asset_value', ' loan_status'],
      dtype='object')

In [26]:
target_encoder = LabelEncoder()
df[TARGET_COL] = target_encoder.fit_transform(df[TARGET_COL])
 

In [27]:
plt.figure(figsize=(5, 4))
sns.countplot(x=TARGET_COL, data=df)
plt.title("Loan Approval Distribution (1=Approved, 0=Rejected)")
plt.savefig("target_distribution.png")
plt.close()
 
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.close()

In [28]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [29]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 

In [32]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}
results = {}

In [33]:
for name, model in models.items():
    # Tree-based models don't need scaling, others do
    if name in ["Decision Tree", "Random Forest"]:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
    else:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
 
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    results[name] = {"accuracy": acc, "roc_auc": auc, "model": model}
 
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
    print(classification_report(y_test, preds, target_names=["Rejected", "Approved"]))
 


=== Logistic Regression ===
Accuracy: 0.9215 | ROC-AUC: 0.9743
              precision    recall  f1-score   support

    Rejected       0.92      0.95      0.94       531
    Approved       0.92      0.87      0.89       323

    accuracy                           0.92       854
   macro avg       0.92      0.91      0.92       854
weighted avg       0.92      0.92      0.92       854


=== Decision Tree ===
Accuracy: 0.9766 | ROC-AUC: 0.9727
              precision    recall  f1-score   support

    Rejected       0.97      0.99      0.98       531
    Approved       0.98      0.96      0.97       323

    accuracy                           0.98       854
   macro avg       0.98      0.97      0.97       854
weighted avg       0.98      0.98      0.98       854


=== Random Forest ===
Accuracy: 0.9801 | ROC-AUC: 0.9983
              precision    recall  f1-score   support

    Rejected       0.98      0.99      0.98       531
    Approved       0.99      0.96      0.97       323

  

In [34]:
best_name = max(results, key=lambda k: results[k]["accuracy"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name} (Accuracy: {results[best_name]['accuracy']:.4f})")
 
if best_name in ["Decision Tree", "Random Forest"]:
    best_preds = best_model.predict(X_test)
else:
    best_preds = best_model.predict(X_test_scaled)
 
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Rejected", "Approved"],
            yticklabels=["Rejected", "Approved"])
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix_best_model.png")
plt.close()



Best model: Random Forest (Accuracy: 0.9801)


In [35]:
if "Random Forest" in results:
    rf_model = results["Random Forest"]["model"]
    importances = pd.Series(rf_model.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=False)
 
    plt.figure(figsize=(8, 5))
    sns.barplot(x=importances.values, y=importances.index)
    plt.title("Feature Importance (Random Forest)")
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    plt.close()
 

In [36]:
joblib.dump(best_model, "best_loan_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
joblib.dump(target_encoder, "target_encoder.pkl")
 
print("\nSaved: best_loan_model.pkl, scaler.pkl, label_encoders.pkl, target_encoder.pkl")
print("Saved plots: target_distribution.png, correlation_heatmap.png,",
      "confusion_matrix_best_model.png, feature_importance.png")


Saved: best_loan_model.pkl, scaler.pkl, label_encoders.pkl, target_encoder.pkl
Saved plots: target_distribution.png, correlation_heatmap.png, confusion_matrix_best_model.png, feature_importance.png


In [37]:
def predict_new_applicant(sample_dict):
    """
    sample_dict: raw feature values matching original (unencoded) columns,
    e.g. {"Gender": "Male", "Married": "Yes", "Dependents": "0", ...}
    """
    sample_df = pd.DataFrame([sample_dict])
 
    for col in categorical_cols:
        if col in sample_df.columns:
            sample_df[col] = label_encoders[col].transform(sample_df[col])
 
    if best_name in ["Decision Tree", "Random Forest"]:
        pred = best_model.predict(sample_df[X.columns])
    else:
        sample_scaled = scaler.transform(sample_df[X.columns])
        pred = best_model.predict(sample_scaled)
 
    return target_encoder.inverse_transform(pred)[0]
 

In [43]:
results = {}

for name, model in models.items():
    # Tree-based models don't need scaling, others do
    if name in ["Decision Tree", "Random Forest"]:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
    else:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    results[name] = {"accuracy": acc, "roc_auc": auc, "model": model}

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
    print(classification_report(y_test, preds, target_names=["Rejected", "Approved"]))


=== Logistic Regression ===
Accuracy: 0.9215 | ROC-AUC: 0.9743
              precision    recall  f1-score   support

    Rejected       0.92      0.95      0.94       531
    Approved       0.92      0.87      0.89       323

    accuracy                           0.92       854
   macro avg       0.92      0.91      0.92       854
weighted avg       0.92      0.92      0.92       854


=== Decision Tree ===
Accuracy: 0.9766 | ROC-AUC: 0.9727
              precision    recall  f1-score   support

    Rejected       0.97      0.99      0.98       531
    Approved       0.98      0.96      0.97       323

    accuracy                           0.98       854
   macro avg       0.98      0.97      0.97       854
weighted avg       0.98      0.98      0.98       854


=== Random Forest ===
Accuracy: 0.9801 | ROC-AUC: 0.9983
              precision    recall  f1-score   support

    Rejected       0.98      0.99      0.98       531
    Approved       0.99      0.96      0.97       323

  